In [0]:
%sql
show catalogs;

catalog
airline_catalog
ext_catalog
main
managed_catalog
my-first-bronze-catalog
samples
system


In [0]:
%sql
SHOW SCHEMAS IN managed_catalog;

databaseName
default
information_schema
managed_schema


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS managed_catalog.managed_schema.expense_volume;

In [0]:
from pyspark.sql.functions import *

users_path = "/Volumes/managed_catalog/managed_schema/expense_volume/users.csv"

expenses_path = "/Volumes/managed_catalog/managed_schema/expense_volume/expenses.csv"


users_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv(users_path)


expenses_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv(expenses_path)


display(users_df)

display(expenses_df)

user_id,user_name,email,monthly_income
U001,Arun,arun@gmail.com,50000
U002,Priya,priya@gmail.com,60000
U003,Karthik,karthik@gmail.com,45000
U004,Meena,meena@gmail.com,55000
U005,Ravi,ravi@gmail.com,70000
U006,Anitha,anitha@gmail.com,40000
U007,Suresh,suresh@gmail.com,65000
U008,Lakshmi,lakshmi@gmail.com,48000
U009,Vijay,vijay@gmail.com,75000
U010,Divya,divya@gmail.com,52000


expense_id,user_id,date,category,amount,description
E001,U001,2026-01-05,Food,2500,Groceries
E002,U001,2026-01-10,Transport,1200,Cab
E003,U001,2026-01-20,Shopping,5000,Clothes
E004,U002,2026-01-03,Food,3000,Restaurant
E005,U002,2026-01-12,Utilities,2500,Electricity
E006,U002,2026-01-25,Shopping,7000,Online Shopping
E007,U003,2026-01-08,Food,1800,Groceries
E008,U003,2026-01-15,Entertainment,6000,Movie
E009,U003,2026-01-28,Transport,1500,Fuel
E010,U004,2026-01-04,Food,2200,Grocery


## Data Cleaning

In [0]:
expenses_df = expenses_df.withColumn(
    "date",
    to_date(col("date"))
)


expenses_df = expenses_df.withColumn(
    "month",
    date_format(col("date"),"yyyy-MM")
)


display(expenses_df)

expense_id,user_id,date,category,amount,description,month
E001,U001,2026-01-05,Food,2500,Groceries,2026-01
E002,U001,2026-01-10,Transport,1200,Cab,2026-01
E003,U001,2026-01-20,Shopping,5000,Clothes,2026-01
E004,U002,2026-01-03,Food,3000,Restaurant,2026-01
E005,U002,2026-01-12,Utilities,2500,Electricity,2026-01
E006,U002,2026-01-25,Shopping,7000,Online Shopping,2026-01
E007,U003,2026-01-08,Food,1800,Groceries,2026-01
E008,U003,2026-01-15,Entertainment,6000,Movie,2026-01
E009,U003,2026-01-28,Transport,1500,Fuel,2026-01
E010,U004,2026-01-04,Food,2200,Grocery,2026-01


In [0]:
expense_details = expenses_df.join(
    users_df,
    "user_id",
    "inner"
)


display(expense_details)

user_id,expense_id,date,category,amount,description,month,user_name,email,monthly_income
U001,E001,2026-01-05,Food,2500,Groceries,2026-01,Arun,arun@gmail.com,50000
U001,E002,2026-01-10,Transport,1200,Cab,2026-01,Arun,arun@gmail.com,50000
U001,E003,2026-01-20,Shopping,5000,Clothes,2026-01,Arun,arun@gmail.com,50000
U002,E004,2026-01-03,Food,3000,Restaurant,2026-01,Priya,priya@gmail.com,60000
U002,E005,2026-01-12,Utilities,2500,Electricity,2026-01,Priya,priya@gmail.com,60000
U002,E006,2026-01-25,Shopping,7000,Online Shopping,2026-01,Priya,priya@gmail.com,60000
U003,E007,2026-01-08,Food,1800,Groceries,2026-01,Karthik,karthik@gmail.com,45000
U003,E008,2026-01-15,Entertainment,6000,Movie,2026-01,Karthik,karthik@gmail.com,45000
U003,E009,2026-01-28,Transport,1500,Fuel,2026-01,Karthik,karthik@gmail.com,45000
U004,E010,2026-01-04,Food,2200,Grocery,2026-01,Meena,meena@gmail.com,55000


In [0]:
monthly_summary = expense_details.groupBy(
    "user_id",
    "user_name",
    "month",
    "monthly_income"
).agg(

    sum("amount")
    .alias("total_expense")

)


display(monthly_summary)

user_id,user_name,month,monthly_income,total_expense
U005,Ravi,2026-01,70000,18500
U010,Divya,2026-01,52000,11000
U004,Meena,2026-01,55000,9700
U002,Priya,2026-01,60000,12500
U007,Suresh,2026-01,65000,14700
U001,Arun,2026-01,50000,8700
U003,Karthik,2026-01,45000,9300
U006,Anitha,2026-01,40000,7000
U009,Vijay,2026-01,75000,23000
U008,Lakshmi,2026-01,48000,7800


In [0]:
monthly_summary = monthly_summary.withColumn(
    "savings",
    col("monthly_income") - col("total_expense")
)


display(monthly_summary)

user_id,user_name,month,monthly_income,total_expense,savings
U005,Ravi,2026-01,70000,18500,51500
U010,Divya,2026-01,52000,11000,41000
U004,Meena,2026-01,55000,9700,45300
U002,Priya,2026-01,60000,12500,47500
U007,Suresh,2026-01,65000,14700,50300
U001,Arun,2026-01,50000,8700,41300
U003,Karthik,2026-01,45000,9300,35700
U006,Anitha,2026-01,40000,7000,33000
U009,Vijay,2026-01,75000,23000,52000
U008,Lakshmi,2026-01,48000,7800,40200


In [0]:
monthly_summary = monthly_summary.withColumn(
    "alert",

    when(
        col("total_expense") >
        col("monthly_income")*0.3,

        "High Spending"

    )

    .otherwise("Normal")

)


display(monthly_summary)

user_id,user_name,month,monthly_income,total_expense,savings,alert
U005,Ravi,2026-01,70000,18500,51500,Normal
U010,Divya,2026-01,52000,11000,41000,Normal
U004,Meena,2026-01,55000,9700,45300,Normal
U002,Priya,2026-01,60000,12500,47500,Normal
U007,Suresh,2026-01,65000,14700,50300,Normal
U001,Arun,2026-01,50000,8700,41300,Normal
U003,Karthik,2026-01,45000,9300,35700,Normal
U006,Anitha,2026-01,40000,7000,33000,Normal
U009,Vijay,2026-01,75000,23000,52000,High Spending
U008,Lakshmi,2026-01,48000,7800,40200,Normal


In [0]:
monthly_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"managed_catalog.managed_schema.expense_monthly_summary"
)

In [0]:
%sql
SELECT * FROM managed_catalog.managed_schema.expense_monthly_summary;

user_id,user_name,month,monthly_income,total_expense,savings,alert
U005,Ravi,2026-01,70000,18500,51500,Normal
U010,Divya,2026-01,52000,11000,41000,Normal
U004,Meena,2026-01,55000,9700,45300,Normal
U002,Priya,2026-01,60000,12500,47500,Normal
U007,Suresh,2026-01,65000,14700,50300,Normal
U001,Arun,2026-01,50000,8700,41300,Normal
U003,Karthik,2026-01,45000,9300,35700,Normal
U006,Anitha,2026-01,40000,7000,33000,Normal
U009,Vijay,2026-01,75000,23000,52000,High Spending
U008,Lakshmi,2026-01,48000,7800,40200,Normal
